# Week1_ex5_material_comparison

Rebuilds the Week1_ex5 disk-and-coil setup with the disk swapped to steel_stainless and then ferrite, solves both, and pulls the ohmic loss out of each so it can be compared against the original cast_iron run.

In [ ]:
import pandas as pd
import os
import ansys.aedt.core

In [ ]:
# quiet the screen logs since this notebook launches AEDT twice back to back
ansys.aedt.core.settings.enable_screen_logs = False

DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)
DT.disable_autosave()

In [ ]:
# everything below is the same build as Week1_ex5.ipynb, just wrapped in a function so it
# can be run once per material without copy-pasting the whole geometry twice. Only the disk's
# material argument actually changes between the two calls at the bottom.
def build_and_solve_disk(material_name, label):
    sol_type = "EddyCurrent"
    M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)
    oDesign = M3D.odesign

    ## Set up output directory for results ##
    proj_name = f"Week1_ex5_{label}"
    base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
    proj_dir = os.path.join(base_dir, proj_name)
    os.makedirs(proj_dir, exist_ok=True)

    ## Save project and apply design name ##
    proj = M3D.oproject
    proj.SaveAs(f"{proj_dir}\\{proj_name}.aedt", True)
    M3D.rename_design(proj_name, save=False)
    M3D.save_project()

    ## Create geometry ##
    M3D.modeler.model_units = "cm"

    # coil - identical to Week1_ex5.ipynb
    helix_start_radius = 15
    helix_radius_change = 3.1
    helix_turns = 4
    helix_segments_per_turn = 12

    oEditor = oDesign.SetActiveEditor("3D Modeler")
    oEditor.CreateUserDefinedPart(
        [
            "NAME:UserDefinedPrimitiveParameters",
            "DllName:=", "SegmentedHelix/PolygonHelix.dll",
            "Version:=", "1.0",
            "NoOfParameters:=", 8,
            "PerformIDTranslation:=", False,
            "Library:=", "syslib",
            [
                "NAME:ParamVector",
                ["NAME:Pair", "Name:=", "PolygonSegments", "Value:=", "4"],
                ["NAME:Pair", "Name:=", "PolygonRadius", "Value:=", "1.5cm"],
                ["NAME:Pair", "Name:=", "StartHelixRadius", "Value:=", f"{helix_start_radius}cm"],
                ["NAME:Pair", "Name:=", "RadiusChange", "Value:=", f"{helix_radius_change}cm"],
                ["NAME:Pair", "Name:=", "Pitch", "Value:=", "0cm"],
                ["NAME:Pair", "Name:=", "Turns", "Value:=", f"{helix_turns}"],
                ["NAME:Pair", "Name:=", "SegmentsPerTurn", "Value:=", f"{helix_segments_per_turn}"],
                ["NAME:Pair", "Name:=", "RightHanded", "Value:=", "1"]
            ]
        ],
        [
            "NAME:Attributes",
            "Name:=", "PolygonHelix1",
            "Flags:=", "",
            "Color:=", "(143 175 143)",
            "Transparency:=", 0,
            "PartCoordinateSystem:=", "Global",
            "UDMId:=", "",
            "MaterialValue:=", "\"vacuum\"",
            "SurfaceMaterialValue:=", "\"\"",
            "SolveInside:=", True,
            "ShellElement:=", False,
            "ShellElementThickness:=", "0cm",
            "ReferenceTemperature:=", "20cel",
            "IsMaterialEditable:=", True,
            "IsSurfaceMaterialEditable:=", True,
            "UseMaterialAppearance:=", False,
            "IsLightweight:=", False
        ])

    coil = M3D.modeler.object_list[-1]
    M3D.assign_material(assignment=coil, material="copper")

    lead_margin = 0.7
    helix_end_radius = helix_start_radius + helix_turns * helix_radius_change
    box2_x = helix_end_radius + lead_margin

    box1 = M3D.modeler.create_box(origin=[14, 0, -2], sizes=[2, 2, -2], name=None, material=None)
    box2 = M3D.modeler.create_box(origin=[box2_x, 0, -2], sizes=[-2, -2, -2], name=None, material=None)

    face1 = box1.faces[0]
    for f in box1.faces:
        if face1.center[0] < f.center[0]:
            face1 = f

    face2 = box2.faces[0]
    for f in box2.faces:
        if face2.center[0] > f.center[0]:
            face2 = f

    face1 = M3D.modeler.create_object_from_face(assignment=face1, non_model=False)
    face2 = M3D.modeler.create_object_from_face(assignment=face2, non_model=False)
    M3D.modeler.connect(assignment=[face1, face2])
    connector = M3D.modeler.object_list[-1]

    vector = [0, 0, 1]
    M3D.modeler.duplicate_along_line(assignment=box1, vector=vector, clones=2, attach=False, is_3d_comp=False, duplicate_assignment=True)
    tmp = M3D.modeler.object_list[-1]
    M3D.modeler.unite(assignment=[box1, tmp], purge=False, keep_originals=False)

    M3D.modeler.duplicate_along_line(assignment=box2, vector=vector, clones=2, attach=False, is_3d_comp=False, duplicate_assignment=True)
    tmp = M3D.modeler.object_list[-1]
    M3D.modeler.unite(assignment=[box2, tmp], purge=False, keep_originals=False)

    M3D.modeler.unite(assignment=[coil, box1, box2, connector], purge=False, keep_originals=False)

    # the disk - material_name is the only thing that changes between the two runs
    disk = M3D.modeler.create_polyhedron(orientation=None, center=[0, 0, 1.5], origin=[41, 0, 1.5],
                                         height=1, num_sides=12, name="Disk", material=material_name)

    ## Assign current excitation ##
    M3D.modeler.section(assignment=coil, plane="YZ", create_new=True, section_cross_object=False)
    coil_terminal = M3D.modeler.sheet_objects[-1]
    M3D.modeler.separate_bodies(assignment=coil_terminal, create_group=False)

    sheets_y = [s.faces[0].center[1] for s in M3D.modeler.sheet_objects]
    positive = [(i, y) for i, y in enumerate(sheets_y) if y > 0]
    terminal_inx = min(positive, key=lambda x: abs(x[1]))[0]
    coil_terminal = M3D.modeler.sheet_objects[terminal_inx]

    for i, s in enumerate(M3D.modeler.sheet_objects):
        if i != terminal_inx:
            M3D.modeler.delete(assignment=s)

    M3D.assign_current(assignment=coil_terminal, amplitude="125A", phase='0deg', solid=True, swap_direction=False, name="I_Coil")

    ## Set up the simulation region ##
    region = M3D.modeler.create_polyhedron(orientation=None, center=[0, 0, -30], origin=[100, 0, -30],
                                           height=60, num_sides=12, name="Region", material=None)

    ## Configure eddy effects ##
    M3D.eddy_effects_on(assignment=coil.name, enable_eddy_effects=False, enable_displacement_current=False)
    M3D.eddy_effects_on(assignment=disk.name, enable_eddy_effects=True, enable_displacement_current=False)

    # setup via script-recording style call, same reasoning as Week1_ex5.ipynb (EddyCurrent -> AC Magnetic
    # rename breaks pyaedt's create_setup()/setups[-1]), same mesh-cap-driven pass/refinement values too
    setup_name = "Setup1"
    oModule = oDesign.GetModule("AnalysisSetup")
    oModule.InsertSetup("EddyCurrent",
        [
            "NAME:Setup1",
            "Enabled:=", True,
            ["NAME:MeshLink", "ImportMesh:=", False],
            "MaximumPasses:=", 15,
            "MinimumPasses:=", 2,
            "MinimumConvergedPasses:=", 1,
            "PercentRefinement:=", 15,
            "SolveFieldOnly:=", False,
            "PercentError:=", 2,
            "SolveMatrixAtLast:=", True,
            "UseNonLinearIterNum:=", False,
            "CacheSaveKind:=", "Delta",
            "ConstantDelta:=", "0s",
            "UseCacheFor:=", ["Freq"],
            "UseIterativeSolver:=", False,
            "RelativeResidual:=", 1E-05,
            "NonLinearResidual:=", 0.0001,
            "RelaxationFactor:=", 1,
            "SmoothBHCurve:=", True,
            "Frequency:=", "500Hz",
            "HasSweepSetup:=", False,
            "UseHighOrderShapeFunc:=", False,
            "ImportMeshForMuLink:=", False,
            "LossAdaptiveCtrl:=", "0.5",
            "UseMuLink:=", False
        ])

    # ohmic loss over the whole disk (no separate calculation sub-volume here, unlike Week4_ex1/ex2's core_cal)
    oModule = oDesign.GetModule("FieldsReporter")
    oModule.AddNamedExpression("disk_ohmic_loss", "Fields",
        [
            "NameOfExpression:=", ["Ohmic_Loss"],
            "EnterVolume:=", [disk.name],
            "Operation:=", ["VolumeValue"],
            "Operation:=", ["Integrate"]
        ])

    oDesign.Analyze(setup_name)

    # get_solution_data() silently fails on EddyCurrent/AC Magnetic setups on this pyaedt version,
    # so read the value back the same way Week4_ex1/ex2 do - push it onto the Field Calculator
    # stack via the raw API and grab it directly
    oModule.CopyNamedExprToStack("disk_ohmic_loss")
    ohmic_loss_val = oModule.GetTopEntryValue("Setup1 : LastAdaptive", ["Freq:=", "500Hz", "Phase:=", "0deg"])[0]
    oModule.CalcStack("clear")

    M3D.save_project()

    return ohmic_loss_val

In [ ]:
results = []
for material_name, label in [("steel_stainless", "steel"), ("ferrite", "ferrite")]:
    loss = build_and_solve_disk(material_name, label)
    results.append({"material": label, "disk_ohmic_loss": loss})

df = pd.DataFrame(results)
df